In [30]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline
)

from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

train = dataset["train"]

print(train)
print(train[0])

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})
{'id': 1, 'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.", 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.", 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.', 'C': 'Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the 

### Q1 Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [32]:
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train = train.map(combine)

print(train[51]["combined_text"])
print("Character Length:", len(train[51]["combined_text"]))

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character Length: 614


### Q2 nitialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [33]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Vocabulary Size:", tokenizer.vocab_size)

Vocabulary Size: 30522


### Q3 Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [34]:
print("[SEP] Token ID:", tokenizer.sep_token_id)

[SEP] Token ID: 102


### Q4 Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [35]:
prompts = list(train["prompt"])

tokenized = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("input_ids shape:", tokenized["input_ids"].shape)

input_ids shape: torch.Size([2000, 128])


## BERT/RoBERTa Architecture & Attention Mechanisms
### Q5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [36]:
hidden_size = 768
heads = 12

head_dim = hidden_size // heads
print("Head dimension:", head_dim)

Head dimension: 64


### Q6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [37]:
model = AutoModel.from_pretrained("bert-base-uncased")

inputs = tokenizer(train[0]["prompt"], return_tensors="pt")

outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


### Q7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).

In [38]:
cls_embedding = outputs.last_hidden_state[0, 0]

sum_first_5 = cls_embedding[:5].sum().item()

print(round(sum_first_5, 4))

-1.2001


### Q8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [39]:
model_attn = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

outputs = model_attn(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print(tokens)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [40]:
fusion_idx = tokens.index("fusion")

attention = outputs.attentions[-1][0, 0]

weight = attention[0, fusion_idx].item()

print(round(weight, 4))

0.1025


### Q9. Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [41]:
mini_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_emb = mini_model.encode(train[0]["prompt"], convert_to_tensor=True)
option_b_emb = mini_model.encode(train[0]["B"], convert_to_tensor=True)

score = util.cos_sim(prompt_emb, option_b_emb)

print(round(score.item(), 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7658


### Q10. Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [42]:
def map3(actual, preds):
    for i, pred in enumerate(preds[:3]):
        if pred == actual:
            return 1 / (i + 1)
    return 0

def mean_map3(actuals, preds):
    return np.mean([map3(a, p) for a, p in zip(actuals, preds)])

In [43]:
tfidf_preds = []

for row in train:
    texts = [row["prompt"]] + [row[x] for x in ["A","B","C","D","E"]]

    vec = TfidfVectorizer().fit_transform(texts)

    sims = cosine_similarity(vec[0:1], vec[1:]).flatten()

    ranked = np.argsort(sims)[::-1]

    labels = np.array(["A","B","C","D","E"])

    tfidf_preds.append(labels[ranked][:3].tolist())

In [44]:
minilm_preds = []

for row in train:
    prompt_emb = mini_model.encode(row["prompt"], convert_to_tensor=True)

    option_embs = mini_model.encode(
        [row[x] for x in ["A","B","C","D","E"]],
        convert_to_tensor=True
    )

    sims = util.cos_sim(prompt_emb, option_embs)[0].cpu().numpy()

    ranked = np.argsort(sims)[::-1]

    labels = np.array(["A","B","C","D","E"])

    minilm_preds.append(labels[ranked][:3].tolist())

In [45]:
actuals = train["answer"]

minilm_map3 = mean_map3(actuals, minilm_preds)

print("MiniLM MAP@3:", round(minilm_map3, 4))

MiniLM MAP@3: 0.4231


In [46]:
count = 0

for actual, tfidf, mini in zip(actuals, tfidf_preds, minilm_preds):
    if actual not in tfidf and actual in mini:
        count += 1

print("Improved Count:", count)

Improved Count: 564


### Q11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [47]:
classifier = pipeline("zero-shot-classification")

result = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ]
)

print(result)
print("Top Score:", round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

### Q12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [48]:
result_multi = classifier(
    train[1]["prompt"],
    candidate_labels=[
        train[1]["A"],
        train[1]["B"],
        train[1]["C"]
    ],
    multi_label=True
)

softmax_sum = sum(result["scores"])
sigmoid_sum = sum(result_multi["scores"])

diff = abs(softmax_sum - sigmoid_sum)

print(round(diff, 4))

0.9995


### Q13. Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 
*


In [49]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = f"""
Question: {train[0]['prompt']}. 
Is the correct answer A: {train[0]['A']} or B: {train[0]['B']}? 
Answer with just the letter A or B.
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated Output:", result)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Generated Output: B
